# MedCLIP-SAMv2 Thigh Segmentation — Sheffield Dataset (Lambda)

Runs [MedCLIP-SAMv2](https://github.com/HealthX-Lab/MedCLIP-SAMv2) zero-shot segmentation
on the 69 Sheffield augmented DICOM volumes (greyscale, used as image proxy).

Pipeline per volume per muscle:
1. Export DICOM multi-frame slices to PNG
2. BiomedCLIP saliency map (text prompt → heatmap per slice)
3. Postprocessing (kmeans → coarse binary mask)
4. SAM refinement (coarse mask → precise boundary)
5. Reassemble PNG masks → 3D NPZ

Data: `~/sheffeld/20440164/Aug_N.dcm`
Output: `~/medclipsamv2_sheffield_segs/Aug_N_medclipsamv2.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medclipsamv2_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os

REPO_DIR = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_DIR = os.path.expanduser('~/mcsam2_env')
VENV_PY  = os.path.join(VENV_DIR, 'bin', 'python')

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/HealthX-Lab/MedCLIP-SAMv2.git', REPO_DIR])

if not os.path.exists(VENV_PY):
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])

def venv_pip(*args):
    subprocess.check_call([VENV_PY, '-m', 'pip'] + list(args))

venv_pip('install', '-q', '--upgrade', 'pip')
venv_pip('install', '-q', 'numpy', 'scikit-learn')
venv_pip('install', '-q', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124')
venv_pip('install', '-q', '-e', os.path.join(REPO_DIR, 'segment-anything'))
venv_pip('install', '-q', 'git+https://github.com/lucasb-eyer/pydensecrf.git')
venv_pip('install', '-q',
    'open_clip_torch', 'opencv-python', 'SimpleITK', 'Pillow', 'pydicom',
    'huggingface_hub', 'transformers<4.46',
    'matplotlib', 'grad-cam', 'pandas', 'tqdm', 'scipy')
print('Dependencies installed.')

In [ ]:
import os, urllib.request

CKPT_DIR  = os.path.join(os.path.expanduser('~/MedCLIP-SAMv2'),
                          'segment-anything', 'sam_checkpoints')
CKPT_FILE = os.path.join(CKPT_DIR, 'sam_vit_b_01ec64.pth')
URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CKPT_FILE):
    print('Downloading SAM ViT-B checkpoint (~375 MB)...')
    urllib.request.urlretrieve(URL, CKPT_FILE)
    print(f'Done ({os.path.getsize(CKPT_FILE) // 1_000_000} MB)')
else:
    print('SAM checkpoint already present')

In [ ]:
import glob, re, shutil, tempfile
import numpy as np
import torch
import cv2
import pydicom
from PIL import Image

IMG_DIR    = os.path.expanduser('~/sheffeld/20440164')
REPO_DIR   = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_PY    = os.path.expanduser('~/mcsam2_env/bin/python')
VENV_DIR   = os.path.expanduser('~/mcsam2_env')
SAM_CKPT   = os.path.join(REPO_DIR, 'segment-anything', 'sam_checkpoints', 'sam_vit_b_01ec64.pth')
SAM_TYPE   = 'vit_b'
OUTPUT_DIR = os.path.expanduser('~/medclipsamv2_sheffield_segs')
SAL_SCRIPT = os.path.join(REPO_DIR, 'saliency_maps', 'generate_saliency_maps.py')
POST_SCRIPT= os.path.join(REPO_DIR, 'postprocessing', 'postprocess_saliency_maps.py')
SAM_SCRIPT = os.path.join(REPO_DIR, 'segment-anything', 'prompt_sam.py')
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs(OUTPUT_DIR, exist_ok=True)

dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm'))
     if '_segmentations' not in f],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
)
print(f'Device: {DEVICE}  |  {len(dcm_files)} volumes')
print('SAL_SCRIPT:', os.path.exists(SAL_SCRIPT))
print('SAM_CKPT:  ', os.path.exists(SAM_CKPT))

In [ ]:
# Thigh muscles matched to Sheffield label set where possible
MUSCLES = [
    ('R_gracilis',    'gracilis muscle right thigh MRI axial cross section'),
    ('L_gracilis',    'gracilis muscle left thigh MRI axial cross section'),
    ('R_sartorius',   'sartorius muscle right thigh MRI axial cross section'),
    ('L_sartorius',   'sartorius muscle left thigh MRI axial cross section'),
    ('R_rectus_femoris',    'rectus femoris muscle right thigh MRI axial cross section'),
    ('L_rectus_femoris',    'rectus femoris muscle left thigh MRI axial cross section'),
    ('R_vastus_lateralis',  'vastus lateralis muscle right thigh MRI axial cross section'),
    ('L_vastus_lateralis',  'vastus lateralis muscle left thigh MRI axial cross section'),
    ('R_vastus_medialis',   'vastus medialis muscle right thigh MRI axial cross section'),
    ('L_vastus_medialis',   'vastus medialis muscle left thigh MRI axial cross section'),
    ('R_vastus_intermedius','vastus intermedius muscle right thigh MRI axial cross section'),
    ('L_vastus_intermedius','vastus intermedius muscle left thigh MRI axial cross section'),
    ('R_semimembranosus',   'semimembranosus muscle right thigh MRI axial cross section'),
    ('L_semimembranosus',   'semimembranosus muscle left thigh MRI axial cross section'),
    ('R_semitendinosus',    'semitendinosus muscle right thigh MRI axial cross section'),
    ('L_semitendinosus',    'semitendinosus muscle left thigh MRI axial cross section'),
    ('R_biceps_femoris',    'biceps femoris muscle right thigh MRI axial cross section'),
    ('L_biceps_femoris',    'biceps femoris muscle left thigh MRI axial cross section'),
    ('R_adductor_magnus',   'adductor magnus muscle right thigh MRI axial cross section'),
    ('L_adductor_magnus',   'adductor magnus muscle left thigh MRI axial cross section'),
]
print(f'{len(MUSCLES)} muscles')

In [ ]:
import glob as _glob
_venv_site = _glob.glob(os.path.join(VENV_DIR, 'lib', 'python3.*', 'site-packages'))
if not _venv_site:
    raise RuntimeError(f'No site-packages in {VENV_DIR}')
VENV_SITE = _venv_site[0]

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH']       = VENV_SITE + ':' + SUBPROCESS_ENV.get('PYTHONPATH', '')
SUBPROCESS_ENV['PYTHONNOUSERSITE'] = '1'
SUBPROCESS_ENV['MPLBACKEND']       = 'Agg'


def run_stage(cmd, stdin_text=None, cwd=None):
    result = subprocess.run(
        cmd, input=stdin_text, text=True, capture_output=True,
        cwd=cwd or REPO_DIR, env=SUBPROCESS_ENV,
    )
    if result.returncode != 0:
        print('STDOUT:', result.stdout[-2000:])
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')


def export_slices_as_png(img_array, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for i in range(img_array.shape[0]):
        sl = img_array[i]
        sl_norm  = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        sl_uint8 = (sl_norm * 255).astype(np.uint8)
        Image.fromarray(np.stack([sl_uint8] * 3, axis=-1)).save(
            os.path.join(out_dir, f'{i}.png')
        )


def load_mask_pngs(mask_dir, num_slices, H, W):
    vol = np.zeros((num_slices, H, W), dtype=np.uint8)
    for i in range(num_slices):
        png_path = os.path.join(mask_dir, f'{i}.png')
        if os.path.exists(png_path):
            m = cv2.imread(png_path, cv2.IMREAD_GRAYSCALE)
            if m is not None:
                if m.shape != (H, W):
                    m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
                vol[i] = (m > 127).astype(np.uint8)
    return vol


print('Helpers defined.')

In [ ]:
for dcm_path in dcm_files:
    idx      = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    out_path = os.path.join(OUTPUT_DIR, f'Aug_{idx}_medclipsamv2.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): Aug_{idx}')
        continue

    print(f'\n═══ Aug_{idx} ═══')
    ds        = pydicom.dcmread(dcm_path)
    img_array = ds.pixel_array.astype(np.float32)   # (D, H, W)
    D, H, W   = img_array.shape
    print(f'  Shape: {img_array.shape}')

    tmp_root  = tempfile.mkdtemp(prefix='mcs2s_')
    all_masks = {}

    try:
        png_dir = os.path.join(tmp_root, 'slices')
        export_slices_as_png(img_array, png_dir)

        for muscle_name, text_prompt in MUSCLES:
            print(f'  [{muscle_name}]')

            sal_dir  = os.path.join(tmp_root, f'sal_{muscle_name}')
            post_dir = os.path.join(tmp_root, f'post_{muscle_name}')
            sam_dir  = os.path.join(tmp_root, f'sam_{muscle_name}')

            # Stage 1: BiomedCLIP saliency
            run_stage(
                [VENV_PY, SAL_SCRIPT,
                 '--input-path',  png_dir,
                 '--output-path', sal_dir,
                 '--val-path',    png_dir,
                 '--model-name',  'BiomedCLIP',
                 '--device',      DEVICE],
                stdin_text=text_prompt + '\n',
            )

            # Stage 2: postprocess → coarse mask
            run_stage(
                [VENV_PY, POST_SCRIPT,
                 '--input-path',  png_dir,
                 '--output-path', post_dir,
                 '--sal-path',    sal_dir,
                 '--postprocess', 'kmeans',
                 '--filter'],
            )

            # Stage 3: SAM refinement
            run_stage(
                [VENV_PY, SAM_SCRIPT,
                 '--input',      png_dir,
                 '--mask-input', post_dir,
                 '--output',     sam_dir,
                 '--model-type', SAM_TYPE,
                 '--checkpoint', SAM_CKPT,
                 '--prompts',    'boxes',
                 '--device',     DEVICE],
            )

            all_masks[muscle_name] = load_mask_pngs(sam_dir, D, H, W)
            print(f'    {int(all_masks[muscle_name].sum()):,} voxels')

        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved → {out_path}')

    except Exception as e:
        print(f'  ERROR on Aug_{idx}: {e}')

    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Output files: {len(results)} / {len(dcm_files)}')
if results:
    s = np.load(results[0])
    for k in sorted(s.files):
        print(f'  {k}: voxels={int(s[k].sum()):,}')